# Reading a protein into mBuild and writing a new PDB file
### Joseph R. Laforet Jr.

A PDB file gives you elements, atom names, and coordinates. You also need the bond orders and the formal
charge of every atom to assign force
field parameters. Those are not always present in PDB files.

Some PDB readers guess this information from interatomic
distances, or omit charges entirely. 

`mbuild.biopolymers` does not guess chemistry. Every residue is matched by atom
name against a template from the wwPDB Chemical Component Dictionary,
and the bonds, bond orders and formal charges come from that template. A
residue not given by a template raises an error.

In [ ]:
from mbuild.biopolymers import Protein

mbuild_protein = Protein("1ubq_protonated.pdb")
print(len(list(mbuild_protein.residues())), "residues,", mbuild_protein.n_particles, "atoms")
print("net formal charge:", mbuild_protein.net_formal_charge)

Formal charges of residues are not something a coordinate reader gives you. They come
from the matched CCD templates, one residue at a time.

In [ ]:
for resnum in (1, 48, 76):
    residue = mbuild_protein.get_residue(resnum, chain_id="A")
    print(f"{residue.name} {resnum:>3}  charge {residue.formal_charge:+d}  "
          f"{residue.atom_formal_charges}")

Bond order is preserved! `to_rdkit` allows you to see that: it
refuses to export a bond whose order is unknown, so the fact that it
returns a sanitized molecule at all is the check.

In [ ]:
from rdkit import Chem
from collections import Counter

rdkit_mol = mbuild_protein.to_rdkit()
print(rdkit_mol.GetNumAtoms(), "atoms, formal charge", Chem.GetFormalCharge(rdkit_mol))

rdkit_mol_bond_orders = Counter(bond.GetBondType() for bond in rdkit_mol.GetBonds())
print(rdkit_mol_bond_orders)

## Handing it to OpenFF

`save_pdb` writes a file with real residue numbers, chain identifiers,
a TER after each chain, and CONECT records only for the bonds that
residue order does not imply, such as disulfides. Peptide bonds are left
implied, because a residue-template reader rejects a CONECT its own
definitions cannot explain.

Ubiquitin has no cysteines, so this file has no CONECT records at all. We are using Ubiquitin for the covalent modification demo later, though.
The lysozyme round trip in
[`evidence/pdb_round_trip.ipynb`](evidence/pdb_round_trip.ipynb) shows
the four disulfides written as CONECT records and read back.

openff-pablo reads that file with no extra arguments.


In [ ]:
from openff.pablo import STD_CCD_CACHE, topology_from_pdb

mbuild_protein.save_pdb("1ubq_FROM_MBUILD.pdb", overwrite=True)

# Note, we can load the mBuild generated protein into OpenFF via Pablo
openff_topology = topology_from_pdb("1ubq_FROM_MBUILD.pdb", residue_library=STD_CCD_CACHE)

openff_molecule = openff_topology.molecule(0)
print(openff_molecule.n_atoms, "atoms, net charge", openff_molecule.total_charge)

openff_molecule_bond_orders = Counter(bond.bond_order for bond in openff_molecule.bonds)
print("RDKit Bond Orders (not kekulized): ", rdkit_mol_bond_orders)
print("OpenFF Bond Orders (kekulized): ", openff_molecule_bond_orders, f"Aromatic: {sum(b.is_aromatic for b in openff_molecule.bonds)}")

print("NOTE: The 5 missing aromatic bonds are from a neutral imidazole in histidine that are not labeled as aromatic by OpenFF's aromaticity model.")

In [ ]:
view = openff_topology.visualize()
view.clear_representations()
view.add_representation("cartoon", color="#990000")
view

That is the round trip for an unmodified protein. The next notebook
showcases how to modify a protein then hand it off to OpenFF.